# Lag-1 TE vs CAMELS Attributes

## What we are doing
Last week we used **weighted-average lag (τ\*)** and **weighted-average TE (TE\*)** across all significant lags. The week before that we used **peak TE** and **peak lag**.

This week we take just one TE value per basin per forcing — the TE at exactly **lag τ = 1 day**. This is the shortest-timescale information transfer from each forcing to streamflow.

For each basin and each forcing (PRCP, SRAD, Tair, VP):

- Take TE(τ=1) **only if** it passed the 200-shuffle significance test.
- If lag 1 is not significant for a forcing, skip that basin–forcing pair. Keep the same basin for the other forcings where lag 1 *is* significant.
- Correlate the lag-1 TE values against 50 CAMELS attributes (Pearson r).

Steps:

1. CSV with TE(τ=1) per basin per forcing — significant only.
2. Scatter plots: each forcing's TE(τ=1) vs each of the 50 attributes.
3. CONUS map: TE(τ=1) per forcing (4 subplots).
4. Ranking tables of attributes by |r|.
5. Comparison with Peak TE and Weighted TE (next task).

## Inputs
- Raw TE values and shuffle results from the older folder:
  `ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)\outputs\summary_csv\TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv`
- 50 CAMELS attributes (same file used last week).

## Outputs (this project folder)
`Lag1_TE vs Attributes\outputs\`
- `summary_csv\` — basin-level lag-1 TE table, correlation tables
- `pdf\` — scatter plots, CONUS maps, ranking tables
- `logs\` — run logs

## Note
Lag-1 TE only captures the fastest possible response. It says nothing about slower processes. So we expect attributes tied to fast runoff (slope, q95, runoff_ratio) to matter more than attributes tied to storage or snowmelt. We will check this against the peak and weighted results in the next task.

## Project setup
Set folder paths and create the output folders. Nothing is computed yet. We just check that the input files we need from last week are reachable.

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import os
import pandas as pd
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth, Reader

In [2]:
# This project folder (where we save outputs)
project_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes")

# Old TE project folder (where we read raw TE and shuffle results from)
te_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)")

# Output subfolders
out_csv  = project_folder / "outputs" / "summary_csv"
out_pdf  = project_folder / "outputs" / "pdf"
out_logs = project_folder / "outputs" / "logs"

for folder in [out_csv, out_pdf, out_logs]:
    folder.mkdir(parents=True, exist_ok=True)

# Where TE results live in the old folder
te_csv_folder = te_folder / "outputs" / "summary_csv"

print("Project folder exists:", project_folder.exists())
print("TE folder exists:     ", te_folder.exists())
print("TE summary_csv exists:", te_csv_folder.exists())
print()
print("Output folders ready:")
print(" ", out_csv)
print(" ", out_pdf)
print(" ", out_logs)

Project folder exists: True
TE folder exists:      True
TE summary_csv exists: True

Output folders ready:
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\summary_csv
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\logs


## Load the TE shuffle file

Read the same TE shuffle file we used last week and the week before. It has TE values and shuffle test results for 671 basins, 4 forcings, lags 1 to 30. For this analysis we will keep only rows where `Lag == 1` **and** `sig == True`.

In [3]:
# TE shuffle file from the old folder
te_path = te_csv_folder / "TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv"

te_df = pd.read_csv(te_path, dtype={"gauge_id": str})
te_df["gauge_id"] = te_df["gauge_id"].str.zfill(8)

print("Shape:", te_df.shape)
print()
print("Columns:", te_df.columns.tolist())
print()
print("First 5 rows:")
print(te_df.head())
print()
print("Unique forcings:", te_df["Source"].unique())
print("Unique lags:    ", sorted(te_df["Lag"].unique()))
print()
print("Significant rows (sig=True):", (te_df["sig"] == True).sum())
print("Non-significant rows       :", (te_df["sig"] == False).sum())
print()

# look at lag 1 only
te_lag1 = te_df[te_df["Lag"] == 1]
print("Lag-1 rows total:", len(te_lag1))
print("Lag-1 significant rows:", (te_lag1["sig"] == True).sum())
print("Lag-1 non-significant rows:", (te_lag1["sig"] == False).sum())
print()
print("Lag-1 significant count by forcing:")
print(te_lag1[te_lag1["sig"] == True].groupby("Source").size())

Shape: (80520, 10)

Columns: ['gauge_id', 'huc_02', 'Source', 'Target', 'Bins', 'Lag', 'TE_obs', 'thr95', 'pval', 'sig']

First 5 rows:
   gauge_id  huc_02 Source Target  Bins  Lag    TE_obs     thr95      pval  \
0  01013500       1   PRCP      Q     5    1  0.002761  0.001187  0.004975   
1  01013500       1   PRCP      Q     5    2  0.001754  0.001171  0.004975   
2  01013500       1   PRCP      Q     5    3  0.001439  0.001204  0.009950   
3  01013500       1   PRCP      Q     5    4  0.000913  0.001174  0.258706   
4  01013500       1   PRCP      Q     5    5  0.001240  0.001064  0.024876   

     sig  
0   True  
1   True  
2   True  
3  False  
4   True  

Unique forcings: ['PRCP' 'SRAD' 'Tair' 'VP']
Unique lags:     [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64

## Extract lag-1 TE per basin per forcing

For each basin and each forcing, take the TE value at **lag 1 only**, and **only if it passed shuffle significance**. Non-significant lag-1 values become NaN. Reshape to wide format so each basin has one row with 4 columns: `TE_lag1_PRCP`, `TE_lag1_SRAD`, `TE_lag1_Tair`, `TE_lag1_VP`.

In [4]:
# keep only lag 1 and significant
te_lag1_sig = te_df[(te_df["Lag"] == 1) & (te_df["sig"] == True)].copy()

print("Lag-1 significant rows:", len(te_lag1_sig))
print()

# pivot to wide: one row per basin, one column per forcing
lag1_wide = te_lag1_sig.pivot(index="gauge_id", columns="Source", values="TE_obs")
lag1_wide.columns = [f"TE_lag1_{c}" for c in lag1_wide.columns]
lag1_wide = lag1_wide.reset_index()

# make sure gauge_id stays zero-padded
lag1_wide["gauge_id"] = lag1_wide["gauge_id"].astype(str).str.zfill(8)

print("Wide table shape:", lag1_wide.shape)
print()
print("Columns:", lag1_wide.columns.tolist())
print()
print("First 5 rows:")
print(lag1_wide.head())
print()
print("Missing values per column (basins with no significant lag-1 TE):")
print(lag1_wide.isna().sum())
print()
print("Summary stats per forcing:")
print(lag1_wide[[c for c in lag1_wide.columns if c != "gauge_id"]].describe().round(5))
print()

# save
out_wide_path = out_csv / "lag1_TE_per_basin.csv"
lag1_wide.to_csv(out_wide_path, index=False)
print("Saved:", out_wide_path)

Lag-1 significant rows: 1891

Wide table shape: (658, 5)

Columns: ['gauge_id', 'TE_lag1_PRCP', 'TE_lag1_SRAD', 'TE_lag1_Tair', 'TE_lag1_VP']

First 5 rows:
   gauge_id  TE_lag1_PRCP  TE_lag1_SRAD  TE_lag1_Tair  TE_lag1_VP
0  01013500      0.002761      0.001838      0.003136    0.002654
1  01022500      0.016681      0.004775      0.008300    0.006192
2  01030500      0.013428      0.003646      0.006792    0.005163
3  01031500      0.003859      0.001481      0.001314    0.001597
4  01047000      0.002277      0.000942      0.001508    0.001330

Missing values per column (basins with no significant lag-1 TE):
gauge_id          0
TE_lag1_PRCP     26
TE_lag1_SRAD    145
TE_lag1_Tair    300
TE_lag1_VP      270
dtype: int64

Summary stats per forcing:
       TE_lag1_PRCP  TE_lag1_SRAD  TE_lag1_Tair  TE_lag1_VP
count     632.00000     513.00000     358.00000   388.00000
mean        0.00536       0.00432       0.00389     0.00311
std         0.00519       0.00324       0.00251     0.00189


### Output check

- 658 basins have at least one significant lag-1 TE. The other 13 basins had no significant lag-1 for any forcing, so they drop out.
- Sample counts match step 2: PRCP 632, SRAD 513, Tair 358, VP 388.
- Mean lag-1 TE by forcing: PRCP = 0.0054, SRAD = 0.0043, Tair = 0.0039, VP = 0.0031. PRCP carries the most information at lag 1, VP the least.
- This order is different from the weighted case, where SRAD and Tair had the highest TE\*. At lag 1 the picture flips because PRCP acts fast and shows up at the shortest lag, while the energy forcings need more time.
- Max PRCP lag-1 TE is 0.034, much higher than the others. A few basins show very strong instantaneous rainfall response, but most are modest (median 0.0034).

## Load the 50 CAMELS attributes and merge with lag-1 TE

Read the same attribute CSV used in the past two weeks (`camels_attributes_combined_671basins.csv`). Merge it with the wide-format lag-1 TE table on `gauge_id`. After this we have one row per basin with lag-1 TE values plus all 50 attributes side by side.

In [5]:
# attribute CSV path (same as last week)
attr_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\camels_attributes_combined_671basins.csv"

attr_df = pd.read_csv(attr_path, dtype={"gauge_id": str})
attr_df["gauge_id"] = attr_df["gauge_id"].str.zfill(8)

print("Attributes shape:", attr_df.shape)
print()
print("First 5 attribute columns:", attr_df.columns.tolist()[:5])
print("Last 5 attribute columns: ", attr_df.columns.tolist()[-5:])
print()

# merge with wide-format lag-1 TE table (left join on lag1_wide keeps only basins with at least 1 significant lag-1)
merged_df = lag1_wide.merge(attr_df, on="gauge_id", how="left")

print("Merged shape:", merged_df.shape)
print()
print("Missing attribute rows after merge (should be 0):",
      merged_df[attr_df.columns[1]].isna().sum())
print()

# save merged file
out_merged_path = out_csv / "lag1_TE_with_attributes.csv"
merged_df.to_csv(out_merged_path, index=False)
print("Saved:", out_merged_path)

Attributes shape: (671, 60)

First 5 attribute columns: ['gauge_id', 'huc_02', 'gauge_name', 'p_mean', 'pet_mean']
Last 5 attribute columns:  ['gvf_diff', 'dom_land_cover_frac', 'dom_land_cover', 'root_depth_50', 'root_depth_99']

Merged shape: (658, 64)

Missing attribute rows after merge (should be 0): 0

Saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\summary_csv\lag1_TE_with_attributes.csv


## Lock the final attribute lists

Two lists:

- **50 numeric attributes** — used for Pearson correlation and scatter plots.
- **2 categorical attributes** (`high_prec_timing`, `low_prec_timing`) — used for boxplots only, same as the past two weeks.

This keeps everything directly comparable with the peak and weighted analyses.

In [6]:
# excluded columns (not used anywhere)
exclude_cols = [
    "gauge_id", "huc_02", "gauge_name", "dom_land_cover",
    "gauge_lat", "gauge_lon",
    "geol_1st_class", "geol_2nd_class",
]

# 2 categorical attributes (for boxplots only)
cat_attrs = ["high_prec_timing", "low_prec_timing"]

# 50 numeric attributes (for correlation + scatter)
attr_list = [c for c in attr_df.columns
             if c not in exclude_cols and c not in cat_attrs]

# 52 attributes total for plotting (50 numeric + 2 categorical)
plot_attr_list = attr_list + cat_attrs

print("Numeric attributes for correlation:", len(attr_list))
print("Categorical attributes for boxplot:", len(cat_attrs))
print("Total attributes for plotting     :", len(plot_attr_list))
print()

# verify all 50 are numeric
non_numeric = attr_df[attr_list].dtypes[
    ~attr_df[attr_list].dtypes.apply(lambda x: pd.api.types.is_numeric_dtype(x))
]
print("Non-numeric in correlation list:", "None" if len(non_numeric) == 0 else list(non_numeric.index))
print()
print("Categorical attributes:", cat_attrs)
print("Excluded columns      :", exclude_cols)

Numeric attributes for correlation: 50
Categorical attributes for boxplot: 2
Total attributes for plotting     : 52

Non-numeric in correlation list: None

Categorical attributes: ['high_prec_timing', 'low_prec_timing']
Excluded columns      : ['gauge_id', 'huc_02', 'gauge_name', 'dom_land_cover', 'gauge_lat', 'gauge_lon', 'geol_1st_class', 'geol_2nd_class']


## Correlate lag-1 TE with the 50 attributes (Pearson)

For each forcing (PRCP, SRAD, Tair, VP) and each of the 50 attributes, compute Pearson r between lag-1 TE and the attribute. NaN values are dropped pairwise. Sample size `n` for each pair is saved alongside the correlation.

Sample sizes are the basins where lag-1 TE was significant for that forcing: PRCP ≈ 632, SRAD ≈ 513, Tair ≈ 358, VP ≈ 388. Tair has the smallest sample, so its correlations are based on fewer basins.

In [8]:
forcings = ["PRCP", "SRAD", "Tair", "VP"]

rows = []
for f in forcings:
    te_col = f"TE_lag1_{f}"
    for attr in attr_list:
        sub = merged_df[[te_col, attr]].dropna()
        n = len(sub)
        if n >= 3:
            r, _ = stats.pearsonr(sub[te_col], sub[attr])
        else:
            r = float("nan")

        rows.append({
            "forcing":   f,
            "attribute": attr,
            "n":         n,
            "r":         r,
        })

corr_df = pd.DataFrame(rows)

# save
out_corr_path = out_csv / "correlation_summary_lag1_TE.csv"
corr_df.to_csv(out_corr_path, index=False)

print("Correlation table shape:", corr_df.shape)
print()
print("First 6 rows:")
print(corr_df.head(6).round(4))
print()
print("Sample size n per forcing (basins used in each correlation):")
print(corr_df.groupby("forcing")["n"].first())
print()
print("Saved:", out_corr_path)
print()

# quick peek: top 5 attributes by |r| per forcing
print("=" * 60)
print("TOP 5 ATTRIBUTES BY |r| FOR EACH FORCING")
print("=" * 60)
for f in forcings:
    sub = corr_df[corr_df["forcing"] == f].copy()
    sub["abs_r"] = sub["r"].abs()
    top5 = sub.sort_values("abs_r", ascending=False).head(5)
    print(f"\n--- {f} (n={int(top5['n'].iloc[0])}) ---")
    for _, row in top5.iterrows():
        print(f"  {row['attribute']:<25} r = {row['r']:+.3f}")

Correlation table shape: (200, 4)

First 6 rows:
  forcing       attribute    n       r
0    PRCP          p_mean  632  0.5845
1    PRCP        pet_mean  632 -0.2066
2    PRCP   p_seasonality  632 -0.3650
3    PRCP       frac_snow  632 -0.0929
4    PRCP         aridity  632 -0.3375
5    PRCP  high_prec_freq  632 -0.3820

Sample size n per forcing (basins used in each correlation):
forcing
PRCP    632
SRAD    513
Tair    358
VP      388
Name: n, dtype: int64

Saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\summary_csv\correlation_summary_lag1_TE.csv

TOP 5 ATTRIBUTES BY |r| FOR EACH FORCING

--- PRCP (n=631) ---
  q95                       r = +0.627
  q_mean                    r = +0.612
  p_mean                    r = +0.585
  runoff_ratio              r = +0.444
  low_prec_freq             r = -0.411

--- SRAD (n=512) ---
  q95                       r = +0.638
  runoff_ratio              r = +0.599
  q_mean  

### Output check

- 200 correlations total = 4 forcings × 50 attributes.
- Sample sizes match step 2: PRCP 632 (one basin lost to NaN in p_mean column gives n=631), SRAD 512, Tair 358, VP 388.
- |r| values reach up to **0.64** (SRAD vs q95), stronger than the weighted case where most top |r| sat between 0.3 and 0.5.

**Top patterns:**

- **PRCP:** q95 (+0.63), q_mean (+0.61), p_mean (+0.59) lead. Wetter, higher-runoff basins show stronger lag-1 rainfall response. Same top attributes as peak and weighted, so PRCP is consistent across all three views.

- **SRAD:** q95 (+0.64), runoff_ratio (+0.60), q_mean (+0.55), slope_mean (+0.49). Streamflow magnitude and slope dominate. Similar story to PRCP.

- **Tair:** Big shift. **frac_snow becomes #1 (+0.47)**, tied with runoff_ratio (+0.47). slope_mean (+0.44) is also high. In peak and weighted analyses, runoff_ratio led for Tair — at lag 1, snow takes over. This is consistent with snowmelt: when temperature crosses thresholds, melt response shows up at very short lags in snowy basins.

- **VP:** Same pattern as Tair, even stronger. **frac_snow #1 (+0.53)**, then elev_mean (+0.42), runoff_ratio (+0.41), slope_mean (+0.40). Snowy high-elevation steep basins show the strongest lag-1 VP response.

**One small surprise:** frac_snow was not in the top 5 for any forcing in the weighted analysis. At lag 1 it jumps to #1 for both Tair and VP. So lag-1 TE is picking up a fast snow-driven signal that gets averaged out in the weighted approach. Worth noting for the comparison task next.

## Quick sanity check

Recompute one Pearson r by hand to confirm the correlation table is right. Use Tair lag-1 vs frac_snow.

In [9]:
# check the data behind Tair r = +0.465 vs frac_snow
check = merged_df[["gauge_id", "TE_lag1_Tair", "frac_snow"]].dropna()
print("Basins used:", len(check))
print()
print(check.head(10))
print()

# recompute
r, _ = stats.pearsonr(check["TE_lag1_Tair"], check["frac_snow"])
print(f"Recomputed r = {r:.4f}")

Basins used: 358

   gauge_id  TE_lag1_Tair  frac_snow
0  01013500      0.003136   0.313440
1  01022500      0.008300   0.245259
2  01030500      0.006792   0.277018
3  01031500      0.001314   0.291836
4  01047000      0.001508   0.280118
5  01052500      0.006612   0.352698
6  01054200      0.002946   0.299642
7  01055000      0.002505   0.306049
8  01057000      0.001128   0.251158
9  01073000      0.002851   0.175032

Recomputed r = 0.4650


### Sanity check

Recomputed Pearson r for Tair lag-1 TE vs frac_snow using 358 basins. Result = **+0.465**, matches the correlation table exactly. Input data looks correct.

## Scatter plots — lag-1 TE vs each attribute

One PDF with 52 pages. Each page = one attribute. Each page has 4 subplots — one per forcing. Points are single color (blue). A regression line and Pearson r are shown on each subplot. The 2 categorical attributes (`high_prec_timing`, `low_prec_timing`) get boxplots instead of scatter plots.

In [11]:
season_order = ["djf", "mam", "jja", "son"]
forcings = ["PRCP", "SRAD", "Tair", "VP"]

out_pdf_path = out_pdf / "Lag1TE_vs_Attributes_52pages.pdf"

with PdfPages(out_pdf_path) as pdf:
    for attr in plot_attr_list:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Lag-1 TE vs {attr}", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):
            te_col = f"TE_lag1_{forcing}"
            sub = merged_df[[attr, te_col]].dropna()

            # CATEGORICAL (boxplot)
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].astype(str).str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s][te_col].values for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                for si, season in enumerate(season_order):
                    y_vals = sub[sub[attr] == season][te_col].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    ax.scatter(si + jitter, y_vals,
                               color="#e41a1c", alpha=0.4, s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}  |  n={len(sub)}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC (scatter + regression)
            else:
                x = sub[attr].values.astype(float)
                y = sub[te_col].values.astype(float)

                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]

                ax.scatter(x, y, color="#e41a1c", alpha=0.5, s=15)

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}", fontsize=10)
                else:
                    ax.set_title(f"{forcing}  |  n={len(x)}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Lag-1 TE", fontsize=8)
            ax.tick_params(labelsize=7)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done. PDF saved to:")
print(out_pdf_path)

Done. PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\Lag1TE_vs_Attributes_52pages.pdf


## CONUS maps — lag-1 TE by forcing

4 subplots, one per forcing. Each basin is colored by its lag-1 TE percentile group (5 groups: 0–20, 20–40, 40–60, 60–80, 80–100th percentile). Basins with no significant lag-1 TE for a forcing are shown as small grey dots.

In [15]:
proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=37.5)

# lag-1 TE bin settings
te_bin_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
te_bin_colors = ["#4a0080", "#66c266", "#ffd700", "#ff8c00", "#8b0000"]
pct_cuts      = [0, 20, 40, 60, 80, 100]

def get_te_bin(val, edges):
    for i in range(len(edges) - 1):
        if edges[i] <= val <= edges[i + 1]:
            return i
    return len(te_bin_labels) - 1

def draw_map_base(ax):
    ax.set_extent([-122, -68, 23, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke")
    ax.add_feature(cfeature.OCEAN, facecolor="#d4eaf7")
    ax.add_feature(cfeature.LAKES, facecolor="#d4eaf7")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.9, edgecolor="black")
    states = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="50m", facecolor="none")
    ax.add_feature(states, edgecolor="grey", linewidth=0.5)

def add_state_names(ax):
    shpfile = natural_earth(resolution="50m", category="cultural",
                            name="admin_1_states_provinces")
    reader = Reader(shpfile)
    for record in reader.records():
        if record.attributes["admin"] != "United States of America":
            continue
        name = record.attributes["name"]
        geom = record.geometry
        cx = geom.centroid.x
        cy = geom.centroid.y
        if cx < -122 or cx > -68 or cy < 23 or cy > 50:
            continue
        ax.text(cx, cy, name, fontsize=5, ha="center", va="center",
                color="dimgrey", transform=ccrs.PlateCarree(),
                fontweight="bold", zorder=4)

def add_title_box(ax, text):
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="grey",
                      boxstyle="round,pad=0.3", alpha=0.85),
            zorder=10)

def add_vertical_legend(ax, colors, labels, title):
    handles = [mpatches.Patch(color=colors[g], label=labels[g])
               for g in range(len(colors))]
    ax.legend(handles=handles,
              loc="upper left",
              bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0,
              fontsize=11,
              title=title,
              title_fontsize=12,
              framealpha=0.9,
              edgecolor="grey",
              handlelength=2.0,
              handleheight=2.0)

# lat/lon for plotting
latlon  = attr_df[["gauge_id", "gauge_lat", "gauge_lon"]].copy()
map_df2 = lag1_wide.merge(latlon, on="gauge_id", how="left")

out_map_path = out_pdf / "CONUS_maps_lag1_TE.pdf"

with PdfPages(out_map_path) as pdf:
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Lag-1 TE by Forcing — 671 CAMELS Basins",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        te_col = f"TE_lag1_{forcing}"
        sub = map_df2[["gauge_lat", "gauge_lon", te_col]].dropna()

        vals  = sub[te_col].values
        edges = [np.percentile(vals, p) for p in pct_cuts]

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(te_bin_labels)):
            mask = sub[te_col].apply(lambda v: get_te_bin(v, edges)) == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=te_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")

        pct_ranges = ["0–20th", "20–40th", "40–60th", "60–80th", "80–100th"]
        te_labels_with_range = [
            f"{te_bin_labels[g]}\n({pct_ranges[g]} percentiles)\n({edges[g]:.4f} – {edges[g+1]:.4f})"
            for g in range(len(te_bin_labels))
        ]
        add_vertical_legend(ax, te_bin_colors, te_labels_with_range, "TE group")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. CONUS map saved to:")
print(out_map_path)

Done. CONUS map saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\CONUS_maps_lag1_TE.pdf


### What the CONUS maps show

Each map colors basins by their lag-1 TE percentile group, separately for each forcing. Dark red = Very High (top 20%), purple = Very Low (bottom 20%). Grey-coloured land has no significant lag-1 basin. The number of basins with significant lag-1 TE is shown in each box: PRCP 632, SRAD 513, Tair 358, VP 388.

**1. The Pacific Northwest is Very High for all four forcings.**
The coastal strip of Washington and Oregon is solid dark red in every panel. These basins show the strongest lag-1 information transfer no matter which forcing we look at. This is the same group that was Very High in the weighted TE\* maps last week, so the pattern is stable across metrics. These are wet, high-runoff, steep basins, which is consistent with the strong positive correlations we saw for q95, runoff_ratio, and slope_mean.

**2. The West in general is stronger than the East.**
For SRAD, Tair, and VP, most of the red and orange points sit in the western half — the Pacific Northwest, the Rockies, and Colorado. The eastern half (Midwest, Northeast, Southeast) has more purple and green points, meaning lower lag-1 TE for these three forcings. So energy and temperature forcings transfer their fastest information mainly in western mountain basins.

**3. PRCP is the most spread out across CONUS.**
The PRCP panel has red and orange points in many regions, not just the West — the Midwest, parts of the South, and the Northeast all show some High and Very High basins. This fits the idea that rainfall drives a quick runoff response almost everywhere, while the energy forcings only do so in certain basin types.

**4. The Northeast shows mixed and often low values.**
For all four forcings, the Northeast cluster (New England, Mid-Atlantic, Ohio) is mostly purple, green, and yellow — Very Low to Medium. Very few Northeast basins are dark red. So lag-1 TE is generally weaker in the Northeast than in the Pacific Northwest, even though both are wet regions. This is worth keeping in mind, because it means "wet" alone does not guarantee strong lag-1 TE — terrain and snow likely matter too.

**5. Snow and elevation show up for Tair and VP.**
The Tair and VP panels have their reddest points in snowy, high-elevation western basins (Cascades, Rockies, Colorado). This matches the correlation table, where frac_snow was the #1 attribute for both Tair (+0.47) and VP (+0.53). The map and the numbers agree.

**Caution.**
These are spatial patterns of information flow at one short timescale (lag 1 day). They are consistent with fast runoff in wet steep basins and fast snowmelt response in snowy basins, but the maps alone do not prove the physical mechanism. We will line this up against the Peak and Weighted maps in the next task to see which basins respond fast versus slow.

## Rank attributes by |r| for lag-1 TE

For each forcing, rank all 50 attributes by absolute Pearson r (strongest first). Save the full ranking as CSV and a top-10 table PDF (one page per forcing). This is directly comparable with the peak and weighted ranking tables.

In [16]:
# build ranked table: sort attributes by |r| within each forcing
ranked_df = corr_df.copy()
ranked_df["abs_r"] = ranked_df["r"].abs()

ranked_list = []
for f in forcings:
    sub = ranked_df[ranked_df["forcing"] == f].sort_values(
        "abs_r", ascending=False).reset_index(drop=True)
    sub["rank"] = sub.index + 1
    ranked_list.append(sub)

ranked_df = pd.concat(ranked_list, ignore_index=True)

# save full ranking CSV
out_rank_csv = out_csv / "ranked_attributes_lag1_TE.csv"
ranked_df.to_csv(out_rank_csv, index=False)
print("Saved:", out_rank_csv)
print()

# build top-10 ranking PDF
ranked_pdf_path = out_pdf / "Ranked_attributes_lag1_TE.pdf"

def build_ranking_page(pdf, title, sub, r_col, n_col):
    n = int(sub[n_col].max())
    fig = plt.figure(figsize=(8.5, 11))
    fig.suptitle(f"{title}\n{sub['forcing'].iloc[0]}  |  n = {n}",
                 fontsize=14, fontweight="bold", y=0.97)

    ax = fig.add_axes([0.08, 0.05, 0.84, 0.85])
    ax.axis("off")

    table_data = [
        [int(row["rank"]), row["attribute"], f"{row[r_col]:+.3f}"]
        for _, row in sub.iterrows()
    ]
    table = ax.table(cellText=table_data,
                     colLabels=["Rank", "Attribute", "r"],
                     loc="upper center", cellLoc="center",
                     colWidths=[0.15, 0.55, 0.20])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.3)

    for j in range(3):
        cell = table[(0, j)]
        cell.set_facecolor("#4a90d9")
        cell.set_text_props(color="white", fontweight="bold")
        cell.set_height(0.022)

    for i in range(1, len(table_data) + 1):
        for j in range(3):
            cell = table[(i, j)]
            cell.set_height(0.020)
            if i <= 10:
                cell.set_facecolor("#fff2cc")
            else:
                cell.set_facecolor("#f7f7f7" if i % 2 == 0 else "white")
            if j == 1:
                cell.set_text_props(ha="left")
            cell.set_edgecolor("grey")

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

with PdfPages(ranked_pdf_path) as pdf:
    for f in forcings:
        sub = ranked_df[ranked_df["forcing"] == f].copy()
        build_ranking_page(pdf, "Ranked Attributes by |r| — Lag-1 TE",
                            sub, "r", "n")

print("Saved PDF:", ranked_pdf_path)
print()

# print top 10 in notebook
print("=" * 60)
print("TOP 10 ATTRIBUTES BY |r| FOR LAG-1 TE")
print("=" * 60)
for f in forcings:
    sub = ranked_df[ranked_df["forcing"] == f].head(10)
    n = int(sub["n"].max())
    print(f"\n--- {f} (n={n}) ---")
    for _, row in sub.iterrows():
        print(f"  {int(row['rank']):2}. {row['attribute']:<25} r = {row['r']:+.3f}")

print("\nDone.")

Saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\summary_csv\ranked_attributes_lag1_TE.csv

Saved PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\Ranked_attributes_lag1_TE.pdf

TOP 10 ATTRIBUTES BY |r| FOR LAG-1 TE

--- PRCP (n=632) ---
   1. q95                       r = +0.627
   2. q_mean                    r = +0.612
   3. p_mean                    r = +0.585
   4. runoff_ratio              r = +0.444
   5. low_prec_freq             r = -0.411
   6. q5                        r = +0.392
   7. high_prec_freq            r = -0.382
   8. p_seasonality             r = -0.365
   9. hfd_mean                  r = -0.360
  10. aridity                   r = -0.337

--- SRAD (n=513) ---
   1. q95                       r = +0.638
   2. runoff_ratio              r = +0.599
   3. q_mean                    r = +0.548
   4. slope_mean             

### Ranking interpretation

The ranking tables sort all 50 attributes by |r| for each forcing. Two clear stories show up — one for the rainfall side (PRCP, SRAD) and one for the temperature/humidity side (Tair, VP).

**PRCP and SRAD — streamflow magnitude dominates.**
For both, the top attributes are q95, q_mean, runoff_ratio, and p_mean (all positive). So wetter, higher-runoff basins show stronger lag-1 TE from rainfall and solar radiation. Precipitation frequency attributes (low_prec_freq, high_prec_freq) are negative — basins with more frequent small events show weaker lag-1 TE. This is the same dominant set we saw in the peak and weighted analyses, so PRCP and SRAD are stable across all three metrics.

**Tair and VP — snow and terrain take over.**
For both, frac_snow jumps to #1 (Tair +0.47, VP +0.53), followed by slope_mean, elev_mean, and runoff_ratio. So snowy, steep, high-elevation basins show the strongest lag-1 TE from temperature and humidity. This makes physical sense: in snow basins, a warm day can trigger melt that reaches the stream quickly, so the fast-timescale (lag 1) signal is strong. baseflow_index also enters the top 10 for both, which is new — it did not lead in the rainfall forcings.

**The key contrast.**
At lag 1, the rainfall forcings care about *how much water* a basin produces (q95, runoff_ratio), while the temperature forcings care about *what kind of basin* it is (snow, slope, elevation). This split is sharper at lag 1 than it was in the weighted analysis, where snow was averaged out and runoff_ratio led for all four forcings.

**stream_elas is consistently negative for Tair and VP.**
stream_elas (streamflow elasticity to precipitation) is negative in the top 5 for both. Basins whose flow responds strongly to rainfall changes tend to show *weaker* lag-1 Tair/VP TE — the rainfall signal likely dominates over the temperature signal in those basins.

**Caution.**
Even the strongest correlation here (SRAD q95 +0.64) leaves most of the variation unexplained. These rankings show which attributes line up with lag-1 TE, not which attributes cause it. The snow link for Tair/VP is the most physically convincing because it has a clear mechanism (melt → fast runoff), but it still needs the comparison with peak/weighted TE to confirm whether it is really a fast-response signal.

## Task 3 — Compare lag-1 TE, Peak TE, and Weighted TE

We compare three TE metrics against the same 50 attributes:

- **Peak TE** — max TE over lags (from the older folder)
- **Weighted TE\*** — TE-weighted average over significant lags (from last week)
- **Lag-1 TE** — TE at exactly lag 1 (this week)

Two comparisons:
1. Attribute rankings side by side (which attributes matter for each metric).
2. CONUS map basin by basin — which basins respond fast (strong lag-1) vs slow (weak lag-1 but strong peak/weighted).

First load the two previous correlation files and confirm their column names so we can line them up correctly.

In [17]:
# Peak TE correlation file (older folder)
peak_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_TE_lag_vs_attributes.csv"

# Weighted TE correlation file (last week's folder)
weighted_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv\correlation_summary_weighted_TE_tau.csv"

peak_df = pd.read_csv(peak_path)
weighted_df = pd.read_csv(weighted_path)

print("Peak file shape:", peak_df.shape)
print("Peak columns:", peak_df.columns.tolist())
print()
print("Weighted file shape:", weighted_df.shape)
print("Weighted columns:", weighted_df.columns.tolist())
print()
print("Lag-1 columns (already in memory as corr_df):", corr_df.columns.tolist())
print()
print("Forcings in each file:")
print("  Peak:    ", sorted(peak_df["forcing"].unique()))
print("  Weighted:", sorted(weighted_df["forcing"].unique()))
print("  Lag-1:   ", sorted(corr_df["forcing"].unique()))

Peak file shape: (200, 6)
Peak columns: ['attribute', 'forcing', 'r_peak_TE', 'n_peak_TE', 'r_peak_lag', 'n_peak_lag']

Weighted file shape: (200, 6)
Weighted columns: ['forcing', 'attribute', 'n_tau', 'r_tau', 'n_TE', 'r_TE']

Lag-1 columns (already in memory as corr_df): ['forcing', 'attribute', 'n', 'r']

Forcings in each file:
  Peak:     ['PRCP', 'SRAD', 'Tair', 'VP']
  Weighted: ['PRCP', 'SRAD', 'Tair', 'VP']
  Lag-1:    ['PRCP', 'SRAD', 'Tair', 'VP']


## Ranking comparison — Peak vs Weighted vs Lag-1 TE

4 pages, one per forcing. Three columns side by side: Peak TE, Weighted TE\*, Lag-1 TE. Each column independently ranked by |r| (strongest first). Top 10 highlighted yellow. This shows whether the same attributes matter across all three TE metrics.

In [18]:
compare_pdf_path = out_pdf / "Compare_Peak_Weighted_Lag1_TE.pdf"

# prepare a ranked copy of each metric for a given forcing
def rank_metric(df, forcing, r_col):
    sub = df[df["forcing"] == forcing].copy()
    sub["abs_r"] = sub[r_col].abs()
    sub = sub.sort_values("abs_r", ascending=False).reset_index(drop=True)
    sub["rank"] = sub.index + 1
    return sub

def draw_column(ax, title, sub, r_col, n_col):
    n = int(sub[n_col].max())
    ax.axis("off")
    ax.set_title(f"{title}\nn = {n}", fontsize=11, fontweight="bold", pad=8)

    data = [[int(row["rank"]), row["attribute"], f"{row[r_col]:+.3f}"]
            for _, row in sub.iterrows()]
    table = ax.table(cellText=data,
                     colLabels=["Rank", "Attribute", "r"],
                     loc="upper center", cellLoc="center",
                     colWidths=[0.18, 0.58, 0.24])
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.15)

    for j in range(3):
        cell = table[(0, j)]
        cell.set_facecolor("#4a90d9")
        cell.set_text_props(color="white", fontweight="bold")
        cell.set_height(0.018)

    for i in range(1, len(data) + 1):
        for j in range(3):
            cell = table[(i, j)]
            cell.set_height(0.016)
            if i <= 10:
                cell.set_facecolor("#fff2cc")
            else:
                cell.set_facecolor("#f7f7f7" if i % 2 == 0 else "white")
            if j == 1:
                cell.set_text_props(ha="left")
            cell.set_edgecolor("grey")

with PdfPages(compare_pdf_path) as pdf:
    for f in forcings:
        peak_sub = rank_metric(peak_df, f, "r_peak_TE")
        weighted_sub = rank_metric(weighted_df, f, "r_TE")
        lag1_sub = rank_metric(corr_df, f, "r")

        fig, axes = plt.subplots(1, 3, figsize=(16, 13))
        fig.suptitle(f"TE Ranking Comparison — {f}",
                     fontsize=15, fontweight="bold", y=0.98)

        draw_column(axes[0], "Peak TE", peak_sub, "r_peak_TE", "n_peak_TE")
        draw_column(axes[1], "Weighted TE*", weighted_sub, "r_TE", "n_TE")
        draw_column(axes[2], "Lag-1 TE", lag1_sub, "r", "n")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Saved comparison PDF:", compare_pdf_path)

Saved comparison PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\Compare_Peak_Weighted_Lag1_TE.pdf


### Ranking comparison interpretation

This compares which attributes matter for each TE metric (Peak, Weighted, Lag-1) across all 4 forcings. The big finding: **PRCP and SRAD stay the same across all three metrics, but Tair and VP change a lot when we switch to lag-1.**

**PRCP — almost identical across all three.**
Top attributes are q95, q_mean, p_mean, runoff_ratio for Peak, Weighted, and Lag-1. The ranking barely moves. So for rainfall, it does not matter which TE metric we use — the same wet/high-runoff attributes lead every time. Lag-1 r values are very close to Peak (q95: Peak +0.624, Lag-1 +0.627), and slightly higher than Weighted (+0.440). PRCP is the most stable forcing.

**SRAD — also very stable.**
q95, runoff_ratio, q_mean, slope_mean lead all three metrics. frac_snow stays around rank 8–10 in every column. Same story as PRCP — switching metric does not change which attributes matter for solar radiation.

**Tair — the interesting one. frac_snow jumps to #1 only at lag-1.**
- Peak TE: runoff_ratio #1, frac_snow #3.
- Weighted TE: high_prec_freq #1, frac_snow drops to #13.
- Lag-1 TE: **frac_snow #1 (+0.47)**.

So snow matters most for Tair only at the shortest timescale. In the weighted average, snow gets buried (rank 13) because it averages over many slow lags. At lag 1, the fast snowmelt signal stands out. baseflow_index and elev_mean also rise at lag-1. This is the clearest example of lag-1 telling a different story than the other two metrics.

**VP — same pattern as Tair, even stronger.**
- Peak TE: frac_snow #1 (+0.48).
- Weighted TE: frac_snow drops to #15, low_prec_freq leads.
- Lag-1 TE: **frac_snow #1 (+0.53)**, elev_mean #2.

VP at lag-1 is dominated by snow and elevation. The weighted version completely loses this — it leads with precipitation frequency instead. So for VP, the choice of metric changes the whole interpretation.

**Main takeaway.**
For the rainfall-side forcings (PRCP, SRAD), all three metrics agree — streamflow magnitude rules. For the temperature-side forcings (Tair, VP), the metric matters a lot: snow and elevation are strongest at lag-1, but get averaged away in the weighted metric. This tells us lag-1 TE captures a **fast snowmelt response** that the other metrics smooth over.

**Caution.**
The Tair and VP lag-1 results use fewer basins (358 and 388) than Peak/Weighted (593 and 577), because many basins had no significant lag-1 TE for those forcings. So part of the ranking shift could come from the smaller,


## Load per-basin Peak TE values

For the ratio map we need the actual peak TE value per basin per forcing, not the correlations. Load the per-basin peak TE summary from the older folder and confirm its columns before computing the ratio.

In [19]:
# per-basin peak TE file from the older folder (same folder as the peak correlation file)
peak_basin_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv")

# list the CSV files in that folder so we can find the right one
print("CSV files in the peak summary folder:")
for f in sorted(peak_basin_folder.glob("*.csv")):
    print("  ", f.name)

CSV files in the peak summary folder:
   all_non_numeric_attributes_vs_forcing_peak_te_summary.csv
   budyko_class_summary_counts_671basins.csv
   budyko_classification_671basins.csv
   camels_attributes_combined_671basins.csv
   categorical_attributes_vs_peak_lag.csv
   categorical_attributes_vs_peak_TE.csv
   comparison_table_all_vs_dominant.csv
   correlation_summary_dominant_forcing.csv
   correlation_summary_TE_lag_vs_attributes.csv
   dominant_forcing_by_budyko_class_671basins.csv
   forcing_rank_summary_counts_671basins.csv
   forcing_ranking_per_basin_671basins.csv
   forcing_wise_peak_te_lag_summary_671basins.csv
   forcing_wise_peak_te_lag_with_attributes_671basins.csv
   numeric_attributes_vs_peak_lag.csv
   numeric_attributes_vs_peak_TE.csv
   peak_lag_by_budyko_class_671basins.csv
   peak_te_by_budyko_class_671basins.csv
   peak_te_vs_numeric_camels_attributes_correlation_all_forcings.csv
   te_summary_main_671basins.csv


## Confirm the per-basin peak TE file

Load `forcing_wise_peak_te_lag_summary_671basins.csv` and check its columns. We need a per-basin, per-forcing peak TE value to build the ratio map.

In [20]:
peak_basin_path = peak_basin_folder / "forcing_wise_peak_te_lag_summary_671basins.csv"

peak_basin_df = pd.read_csv(peak_basin_path, dtype={"gauge_id": str})
peak_basin_df["gauge_id"] = peak_basin_df["gauge_id"].str.zfill(8)

print("Shape:", peak_basin_df.shape)
print()
print("Columns:", peak_basin_df.columns.tolist())
print()
print("First 8 rows:")
print(peak_basin_df.head(8))
print()
if "Source" in peak_basin_df.columns:
    print("Unique forcings:", peak_basin_df["Source"].unique())

Shape: (671, 9)

Columns: ['gauge_id', 'PRCP_peak_te', 'SRAD_peak_te', 'Tair_peak_te', 'VP_peak_te', 'PRCP_peak_lag', 'SRAD_peak_lag', 'Tair_peak_lag', 'VP_peak_lag']

First 8 rows:
   gauge_id  PRCP_peak_te  SRAD_peak_te  Tair_peak_te  VP_peak_te  \
0  01013500      0.002761      0.003063      0.004888    0.002685   
1  01022500      0.016681      0.004775      0.008300    0.006192   
2  01030500      0.013428      0.004072      0.007460    0.005949   
3  01031500      0.003859      0.001481      0.001351    0.001597   
4  01047000      0.002277      0.001389      0.001541    0.001330   
5  01052500      0.003615      0.005471      0.006612    0.004386   
6  01054200      0.002115      0.002888      0.003738    0.003328   
7  01055000      0.003136      0.003516      0.002714    0.003184   

   PRCP_peak_lag  SRAD_peak_lag  Tair_peak_lag  VP_peak_lag  
0              1             13             14           22  
1              1              1              1            1  
2         

## Ratio map — Lag-1 TE / Peak TE by forcing

For each basin and forcing, compute ratio = lag-1 TE / peak TE. Both come from the same significant TE values.

- Ratio near 1.0 → fast responder. Most of the basin's information transfer is already at lag 1; lag 1 *is* the peak.
- Low ratio (near 0) → slow responder. Lag-1 TE is weak compared to the peak, which sits at a longer lag.

We only compute the ratio where lag-1 TE is significant (the basins already in `lag1_wide`). Peak TE is always ≥ lag-1 TE, so the ratio stays between 0 and 1.

In [21]:
proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=37.5)

# merge lag-1 TE (significant only) with per-basin peak TE
ratio_df = lag1_wide.merge(peak_basin_df, on="gauge_id", how="left")

# compute ratio per forcing: lag-1 TE / peak TE
for f in forcings:
    lag1_col = f"TE_lag1_{f}"
    peak_col = f"{f}_peak_te"
    ratio_df[f"ratio_{f}"] = ratio_df[lag1_col] / ratio_df[peak_col]

# add lat/lon for plotting
ratio_df = ratio_df.merge(latlon, on="gauge_id", how="left")

# quick check on ratio range
print("Ratio summary per forcing (should be between 0 and 1):")
for f in forcings:
    col = f"ratio_{f}"
    sub = ratio_df[col].dropna()
    print(f"  {f}: n={len(sub)}, min={sub.min():.3f}, max={sub.max():.3f}, mean={sub.mean():.3f}")
print()

# ratio color bins (fast vs slow)
ratio_bin_edges  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0001]
ratio_bin_labels = ["0.0–0.2 (slow)", "0.2–0.4", "0.4–0.6", "0.6–0.8", "0.8–1.0 (fast)"]
ratio_bin_colors = ["#4a0080", "#66c266", "#ffd700", "#ff8c00", "#8b0000"]

def get_ratio_bin(val):
    for i in range(len(ratio_bin_edges) - 1):
        if ratio_bin_edges[i] <= val < ratio_bin_edges[i + 1]:
            return i
    return len(ratio_bin_labels) - 1

def draw_map_base(ax):
    ax.set_extent([-122, -68, 23, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke")
    ax.add_feature(cfeature.OCEAN, facecolor="#d4eaf7")
    ax.add_feature(cfeature.LAKES, facecolor="#d4eaf7")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.9, edgecolor="black")
    states = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="50m", facecolor="none")
    ax.add_feature(states, edgecolor="grey", linewidth=0.5)

def add_state_names(ax):
    shpfile = natural_earth(resolution="50m", category="cultural",
                            name="admin_1_states_provinces")
    reader = Reader(shpfile)
    for record in reader.records():
        if record.attributes["admin"] != "United States of America":
            continue
        name = record.attributes["name"]
        geom = record.geometry
        cx, cy = geom.centroid.x, geom.centroid.y
        if cx < -122 or cx > -68 or cy < 23 or cy > 50:
            continue
        ax.text(cx, cy, name, fontsize=5, ha="center", va="center",
                color="dimgrey", transform=ccrs.PlateCarree(),
                fontweight="bold", zorder=4)

def add_title_box(ax, text):
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="grey",
                      boxstyle="round,pad=0.3", alpha=0.85),
            zorder=10)

def add_vertical_legend(ax, colors, labels, title):
    handles = [mpatches.Patch(color=colors[g], label=labels[g])
               for g in range(len(colors))]
    ax.legend(handles=handles, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
              fontsize=10, title=title, title_fontsize=11,
              framealpha=0.9, edgecolor="grey",
              handlelength=2.0, handleheight=2.0)

out_ratio_path = out_pdf / "CONUS_maps_lag1_over_peak_ratio.pdf"

with PdfPages(out_ratio_path) as pdf:
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Lag-1 TE / Peak TE Ratio by Forcing — Fast vs Slow Responders",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        col = f"ratio_{forcing}"
        sub = ratio_df[["gauge_lat", "gauge_lon", col]].dropna()

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(ratio_bin_labels)):
            mask = sub[col].apply(get_ratio_bin) == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=ratio_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")
        add_vertical_legend(ax, ratio_bin_colors, ratio_bin_labels,
                            "Lag-1 / Peak ratio")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. Ratio map saved to:")
print(out_ratio_path)

Ratio summary per forcing (should be between 0 and 1):
  PRCP: n=632, min=0.318, max=1.000, mean=0.978
  SRAD: n=513, min=0.493, max=1.000, mean=0.950
  Tair: n=358, min=0.434, max=1.000, mean=0.851
  VP: n=388, min=0.554, max=1.000, mean=0.907

Done. Ratio map saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\CONUS_maps_lag1_over_peak_ratio.pdf


### Ratio map interpretation

The ratio = lag-1 TE / peak TE. A value near 1.0 means lag 1 is already the peak (fast responder). A low value means the peak sits at a longer lag (slow responder).

**The main result: almost every basin is a fast responder.**
Mean ratio is very high for all forcings: PRCP 0.98, SRAD 0.95, VP 0.91, Tair 0.85. The map is almost all dark red (0.8–1.0). So for the basins included here, lag-1 TE is already at or very close to the peak. The information transfer happens fast — within one day — for most basins and most forcings.

**Forcing order makes sense.**
PRCP has the highest mean ratio (0.98) — rainfall response is the fastest and most consistently peaks at lag 1. Tair has the lowest (0.85) — temperature is the most likely to peak at a longer lag (snowmelt, seasonal ET), so more Tair basins show orange/yellow. This matches the physical expectation.

**The few slow responders.**
The orange and yellow points (lower ratio) are scattered, with a small cluster in the Northeast and Upper Midwest for Tair and VP. These are basins where the temperature signal builds up over several days before peaking — consistent with snow and storage delaying the response.

### One honest caution

This ratio is biased toward 1.0 by how we built it. We only kept basins where lag-1 TE passed the shuffle test. A basin only has a significant lag-1 value if its lag-1 TE is already strong — and a strong lag-1 TE is, by definition, likely to be at or near the peak. So we more or less guaranteed a high ratio for every basin in the sample.

The basins that are genuinely slow responders — strong peak at lag 10, near-zero TE at lag 1 — were mostly dropped in the first place, because their lag-1 TE failed significance. They are the grey/missing basins, not the dark red ones. So the map answers "among basins with a significant lag-1 signal, how fast do they peak?" (answer: very fast), but it does not show the full fast-vs-slow split across all 671 basins.

### What we did about it

Two options:

1. Keep this ratio map. It is still a valid statement, just narrower than it looks.
2. Add a second map using peak lag across all basins, which directly answers the fast-vs-slow question without the bias.

We did both. The peak-lag map (next) uses all 671 basins with no significance filter, so the slow responders are not dropped.

## Peak-lag map — fast vs slow responders (all basins)

The ratio map only used basins with a significant lag-1 TE, which biased it toward fast responders. To see the honest full picture, we map **peak lag** (the lag where TE is strongest) for all basins, with no significance filter.

- Short peak lag (1–2 days) → fast responder.
- Long peak lag (19–30 days) → slow responder.

This directly answers "which basins respond fast vs slow across CONUS" without dropping the slow ones.

In [24]:
# peak lag bins (days) — same groups as the weighted notebook
lag_bin_edges  = [0, 2, 6, 11, 18, 30]
lag_bin_labels = ["1–2 (fast)", "3–6", "7–11", "12–18", "19–30 (slow)"]
lag_bin_colors = ["#8b0000", "#ff8c00", "#ffd700", "#66c266", "#4a0080"]

def get_lag_bin(val):
    for i in range(len(lag_bin_edges) - 1):
        if lag_bin_edges[i] < val <= lag_bin_edges[i + 1]:
            return i
    return len(lag_bin_labels) - 1

peaklag_df = peak_basin_df.merge(latlon, on="gauge_id", how="left")

out_peaklag_path = out_pdf / "CONUS_maps_peak_lag_fast_slow.pdf"

with PdfPages(out_peaklag_path) as pdf:
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Peak Lag by Forcing — Fast vs Slow Responders (All 671 Basins)",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        col = f"{forcing}_peak_lag"
        sub = peaklag_df[["gauge_lat", "gauge_lon", col]].dropna()
        groups = sub[col].apply(get_lag_bin)

        draw_map_base(ax)
        add_state_names(ax)

        # count per bin for this forcing
        counts = [int((groups == g).sum()) for g in range(len(lag_bin_labels))]

        # plot points
        for g in range(len(lag_bin_labels)):
            mask = groups == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=lag_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")

        # legend labels with basin count appended
        labels_with_n = [f"{lag_bin_labels[g]}  (n = {counts[g]})"
                         for g in range(len(lag_bin_labels))]
        add_vertical_legend(ax, lag_bin_colors, labels_with_n, "Peak lag (days)")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. Peak-lag map saved to:")
print(out_peaklag_path)

# quick fast/slow count per forcing
print("Peak-lag group counts per forcing:")
for f in forcings:
    col = f"{f}_peak_lag"
    sub = peaklag_df[col].dropna()
    counts = sub.apply(get_lag_bin).value_counts().sort_index()
    print(f"\n--- {f} (n={len(sub)}) ---")
    for g in range(len(lag_bin_labels)):
        c = counts.get(g, 0)
        print(f"  {lag_bin_labels[g]:<15}: {c}")

Done. Peak-lag map saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\CONUS_maps_peak_lag_fast_slow.pdf
Peak-lag group counts per forcing:

--- PRCP (n=671) ---
  1–2 (fast)     : 598
  3–6            : 16
  7–11           : 14
  12–18          : 20
  19–30 (slow)   : 23

--- SRAD (n=671) ---
  1–2 (fast)     : 434
  3–6            : 43
  7–11           : 31
  12–18          : 53
  19–30 (slow)   : 110

--- Tair (n=671) ---
  1–2 (fast)     : 130
  3–6            : 85
  7–11           : 101
  12–18          : 99
  19–30 (slow)   : 256

--- VP (n=671) ---
  1–2 (fast)     : 219
  3–6            : 78
  7–11           : 93
  12–18          : 98
  19–30 (slow)   : 183


### Peak-lag map interpretation (all 671 basins, no significance filter)

Now that we use peak lag for all basins, the fast-vs-slow split is real and not biased. The four forcings fall on a clear fast-to-slow gradient.

**PRCP — overwhelmingly fast.**
598 of 671 basins (89%) peak at lag 1–2 days. The map is almost solid dark red everywhere. Rainfall reaches streamflow within a day or two in nearly every basin across CONUS. Only 23 basins are slow (19–30 days), scattered with no strong region. This confirms PRCP is the fast forcing — and here the result is honest because we did not drop any basins.

**SRAD — mostly fast but with a clear slow tail.**
434 basins (65%) are fast, but 110 (16%) are slow. The slow SRAD basins cluster in the **Northeast and Upper Midwest** (purple points around New York, Pennsylvania, the Great Lakes). So solar radiation drives a quick response in most of CONUS, but in the cooler, wetter Northeast it acts more slowly.

**Tair — mostly slow.**
Only 130 basins (19%) are fast, while 256 (38%) are slow. This is the opposite of PRCP. The slow Tair basins dominate the **eastern half** — Northeast, Ohio Valley, Southeast. Temperature information takes weeks to reach streamflow in most basins, consistent with slow processes like snowmelt timing and seasonal ET rather than a direct daily response. The fast Tair basins are mostly in the **Pacific Northwest and mountain West**, where snowmelt can respond quickly to a warm day.

**VP — in between Tair and SRAD.**
219 fast (33%) and 183 slow (27%). VP sits between the fast rainfall forcing and the slow temperature forcing. Slow VP basins again cluster in the **Northeast and Great Lakes**.

**The big pattern: a geographic fast-vs-slow divide.**
For the energy forcings (SRAD, Tair, VP), the **West responds fast and the Northeast/Upper Midwest responds slow.** The same Northeast cluster that showed weak lag-1 TE earlier shows up here as the slow-responding region. So the two analyses agree: Northeast basins are slow responders for the energy forcings, which is why their lag-1 TE was weak and often not significant. This is the honest version of the fast-vs-slow story the ratio map could not show.

**Why this matters next to the ratio map.**
The ratio map said "lag-1 basins peak fast" (true but narrow). This peak-lag map shows the real split: PRCP is fast everywhere, but the energy forcings have a large slow population concentrated in the Northeast. The slow basins are exactly the ones the ratio map dropped. Both maps are now in the notebook, and together they give the complete picture.